In [ ]:
import pandas as pd
import numpy as np

print("="*70)
print(" JANUARY 2026: DATA PREPROCESSING")
print("="*70)

print("\n Loading raw datasets...")
duo_raw = pd.read_csv('ml_dataset_duo_mw_reuters_full.csv')
trio_raw = pd.read_csv('ml_dataset_trio_rtt_mw_reuters_full.csv')

print(f"   Duo raw shape: {duo_raw.shape}")
print(f"   Trio raw shape: {trio_raw.shape}")

def process_dataset(df, dataset_name):
    """Reshape wide format to long format and filter for true parallel coverage"""
    print(f"\n Processing {dataset_name.upper()} dataset...")
    
    chunk_size = 8
    num_sources = df.shape[1] // chunk_size
    
    cols = ['ID', 'TITLE', 'URL', 'PUBLISHER', 'CATEGORY', 'STORY', 'HOSTNAME', 'TIMESTAMP']
    
    chunks = []
    for i in range(0, df.shape[1], chunk_size):
        chunk = df.iloc[:, i:i+chunk_size].copy()
        chunk.columns = cols
        chunks.append(chunk)
    
    long_df = pd.concat(chunks, ignore_index=True)
    
    long_df = long_df.dropna(subset=['TITLE', 'PUBLISHER', 'STORY'])
    long_df['TITLE'] = long_df['TITLE'].astype(str).str.strip()
    long_df = long_df[long_df['TITLE'].str.len() > 10]  
    long_df['STORY'] = long_df['STORY'].astype(str).str.strip()
    
    clean_df = long_df[['TITLE', 'PUBLISHER', 'STORY']].copy()
    
    story_counts = clean_df.groupby('STORY')['PUBLISHER'].nunique()
    required_sources = 2 if dataset_name == 'duo' else 3
    valid_stories = story_counts[story_counts == required_sources].index
    
    clean_df = clean_df[clean_df['STORY'].isin(valid_stories)].reset_index(drop=True)
    
    output_file = f'clean_{dataset_name}_data.csv'
    clean_df.to_csv(output_file, index=False)
    
    print(f"    {dataset_name.upper()} processed!")
    print(f"    Total articles: {len(clean_df):,}")
    print(f"    Sources: {clean_df['PUBLISHER'].unique()}")
    print(f"    Unique stories: {clean_df['STORY'].nunique():,}")
    print(f"    Saved to: {output_file}")
    
    print(f"\n    Sample data:")
    print(clean_df.head(3).to_string())
    
    return clean_df

print("\n" + "="*70)
duo_clean = process_dataset(duo_raw, 'duo')
trio_clean = process_dataset(trio_raw, 'trio')

print("\n" + "="*70)
print(" PREPROCESSING COMPLETE!")
print("="*70)
print(f"\n Summary:")
print(f"   Duo:   {len(duo_clean):,} articles | {duo_clean['STORY'].nunique():,} stories")
print(f"   Trio:  {len(trio_clean):,} articles | {trio_clean['STORY'].nunique():,} stories")
print(f"\n Files ready for February baseline experiments:")
print(f"   - clean_duo_data.csv")
print(f"   - clean_trio_data.csv")

📅 JANUARY 2026: DATA PREPROCESSING

📂 Loading raw datasets...
   Duo raw shape: (3073, 8)
   Trio raw shape: (2946, 8)


🔄 Processing DUO dataset...
   ✅ DUO processed!
   📊 Total articles: 3,073
   📰 Sources: ['Reuters' 'MarketWatch']
   🔗 Unique stories: 633
   📁 Saved to: clean_duo_data.csv

   📝 Sample data:
                                                                         TITLE PUBLISHER                          STORY
0                                 Europe reaches crunch point on banking union   Reuters  dPhGU51DcrolUIMxbRm0InaHGA2XM
1  ECB FOCUS-Stronger euro drowns out ECB's message to keep rates low for  ...   Reuters  dPhGU51DcrolUIMxbRm0InaHGA2XM
2  REFILE-Bad loan triggers key feature in ECB bank test announcement- sources   Reuters  dPhGU51DcrolUIMxbRm0InaHGA2XM

🔄 Processing TRIO dataset...
   ✅ TRIO processed!
   📊 Total articles: 2,946
   📰 Sources: ['RTT News' 'Reuters' 'MarketWatch']
   🔗 Unique stories: 377
   📁 Saved to: clean_trio_data.csv

   📝 Sample data